In [1]:
#load imports

%load_ext autoreload
%autoreload 2

import numpy as np
import src.demo as demo
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib
from src.ndf_interface import NDFPartInterface

pybullet build time: May 20 2022 19:45:31


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [11]:
#load pointclouds, interaction points, meshes, and program

#Load the files and camera information
save_name = '/home/rthomp12/fewshot/red_mug_thin_rack_test/init_scene_pcls.npz'
scene_pcls = np.load(save_name)
scene_pcls = {k: scene_pcls[k] for k in scene_pcls.keys()}

#Load interaction points and 
demo_file = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/interaction_points.pkl'


warps = np.load("/home/rthomp12/fewshot/blue_mug_thin_rack_demo/initial_scene_warps.npz", allow_pickle=True)
warps = {k: warps[k] for k in warps.keys()}

child_params = warps['child_params'].item()
parent_params = warps['parent_params'].item()
child_reconstruction = warps['child_reconstructions'].item()

In [1]:
#set the kind of demo
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Mug v Rack
parent_part_names = ['trunk', 'branch']
child_part_names = ['cup', 'handle']

parent_part_model_files = {'trunk': '/home/rthomp12/fewshot/part_based_warp_models/trunk_dict_20240412-042732', 
                           'branch': '/home/rthomp12/fewshot/part_based_warp_models/branch_dict_20240412-042732'}

child_part_model_files = {'cup': '/home/rthomp12/fewshot/part_based_warp_models/cup_dict_20240202-160637',
                          'handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20240202-160637'}
demo_pair = (('cup', 'branch'), ('handle', 'branch')) #TODO: Save and load from file


# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Teapot v Mug
# parent_part_names = ['cup', 'handle']
# child_part_names = 

# parent_part_model_files = {'cup': '/home/rthomp12/fewshot/part_based_warp_models/cup_dict_20240202-160637',
#                           'handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20240202-160637'}
# child_part_model_files = 

# demo_pair = (('cup', 'branch'), ('handle', 'branch')) #TODO: Save and load from file


#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Bowl v Mug
# parent_part_names = ['cup', 'handle']
# child_part_names = 

# parent_part_model_files = {'cup': '/home/rthomp12/fewshot/part_based_warp_models/cup_dict_20240202-160637',
#                           'handle': '/home/rthomp12/fewshot/part_based_warp_models/handle_dict_20240202-160637'}
# child_part_model_files = 

# demo_pair = (('cup', 'branch'), ('handle', 'branch')) #TODO: Save and load from file


parent_part_models = {part: CanonPart.from_pickle(parent_part_model_files[part]) for part in parent_part_names}
child_part_models = {part: CanonPart.from_pickle(child_part_model_files[part]) for part in child_part_names}
child_parts = {name: scene_pcls[name] for name in child_part_names}
parent_parts = {name: scene_pcls[name] for name in parent_part_names}

interface = NDFPartInterface(
        canon_source_parts_paths = child_part_model_files,
        canon_target_parts_paths = parent_part_model_files,
        source_part_names = child_part_names,
        target_part_names = parent_part_names,)


# parent_part_models = {part: CanonPart.from_pickle(parent_part_model_files[part]) for part in parent_part_names}
# child_part_models = {part: CanonPart.from_pickle(child_part_model_files[part]) for part in child_part_names}
# child_parts = {name: scene_pcls[name] for name in child_part_names}
# parent_parts = {name: scene_pcls[name] for name in parent_part_names}

# interface = NDFPartInterface(
#         canon_source_parts_paths = child_part_model_files,
#         canon_target_parts_paths = parent_part_model_files,
#         source_part_names = child_part_names,
#         target_part_names= parent_part_names,)


NameError: name 'CanonPart' is not defined

In [15]:
trans_predicted, constraint_pcl = interface.infer_relpose(
        child_parts, parent_parts, demo_pair, se3=True, knn_pkl=demo_file, return_constraint_pcl=True
    )
print(trans_predicted)

trunk pose: [ 0.77848685 -0.19201379  0.18236702]
branch pose: [ 0.75945402 -0.15684513  0.23049404]

(12808,)
final best idx: 24
final cost: 0.02003300003707409
initial_transform: [[-0.75860752 -0.33109732 -0.56114988]
 [ 0.56386449 -0.76514788 -0.31081434]
 [-0.32645284 -0.55219859  0.76714096]]
[[-0.90571309  0.26267096  0.33269772  1.25429104]
 [-0.42388208 -0.566395   -0.70676778  0.13976317]
 [ 0.00279095 -0.78115342  0.62433287  0.16869756]
 [ 0.          0.          0.          1.        ]]


In [18]:
#visualize with pcd and mesh
transformed_scene_pcls = {}
for child_part in child_part_names:

    transformed_scene_pcls[child_part] = utils.transform_pcd(scene_pcls[child_part],
                                                 trans_predicted,
                                               )
for parent_part in parent_part_names:
    transformed_scene_pcls[parent_part] = scene_pcls[parent_part]
                                                 
#     reconstructions[f'reconstructed_{child_part}'] = \
#         child_part_models[child_part].to_transformed_pcd(child_params[child_part])
    
viz_utils.show_pcds_plotly(transformed_scene_pcls | {'constraint': constraint_pcl})